# 5.7. Predicting House Prices on Kaggle

Let's put our deep learning skills to the test by training a machine learning model on the [Ames Housing Dataset](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset).

Our data pre-processing pipeline is described below.

For the features:

1. Remove the redundant ID column
1. Compute the mean for each column of numerical data, skipping NA values from the mean computation
1. Replace NA values in numerical data with the mean value from their corresponding column
1. Apply StandardScaler to numerical columns to have zero mean and unit variance
1. Apply one-hot encoding to categorical \(non-numerical\) columns with the following rules:
    1. Treat NA values as its own distinct category
    1. Remove the 1st category from the generated one-hot encoding to avoid collinearity

For the labels:

1. Apply `log1p` transformation to the sale price
1. Apply StandardScaler to \(1\) so the transformed labels have zero mean and unit variance

The dataset we expect to get after pre-processing as below.

1. 1460 training samples
1. Approximately 331 input features - generated from the original 80 input features after applying one-hot encoding
1. 1 output label corresponding to the house price

Since we have limited, high-dimensional data, we'll use $k$-fold cross validation with $k = 10$.

We'll train our neural network with the architecture below over 100 epochs with a batch size of $2 ^ 6 = 64$.

1. 1st hidden layer: 331 input channels, 128 output channels, ReLU activation
1. Dropout layer with `p=0.2`
1. 2nd hidden layer: 128 input channels, 64 output channels, ReLU activation
1. Final fully connected layer with 64 input channels and 1 output channel. The output channel is the \(transformed\) predicted sale price

With the outputs from our model, we plan to apply the following inverse transformations to obtain meaningful predicted housing sale prices in USD.

1. Apply the inverse of StandardScaler to recover the mean and variance of the `log1p` transformed housing prices based on the training data
1. Apply `expm1` to the results in \(1\) to undo the `log1p` transformation and obtain the actual predicted housing price in USD

Our choice of loss function, optimizer and hyperparameters below.

1. Loss function: MSE loss on the \(transformed\) predictions
1. Optimizer: minibatch SGD with batches of size 64
1. Learning rate: 0.01
1. Weight decay: `1e-4`
1. Momentum: `0.9`

In [1]:
import mindspore

mindspore.set_device(device_target='Ascend', device_id=0)
mindspore.run_check()

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

MindSpore version:  2.8.0
The result of multiplication calculation is correct, MindSpore has been installed on platform [Ascend] successfully!


## 5.7.1. Downloading Data

We'll download a copy of the Ames housing dataset from [D2L](https://d2l.ai/)-owned S3 bucket\(s\).

In [2]:
import os

dataset_dir = 'data/ames/'
os.makedirs(dataset_dir, exist_ok=True)

In [3]:
import urllib.request

train_ds_link = 'http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_train.csv'
test_ds_link = 'http://d2l-data.s3-accelerate.amazonaws.com/kaggle_house_pred_test.csv'
train_ds_path = os.path.join(dataset_dir, 'kaggle_house_pred_train.csv')
test_ds_path = os.path.join(dataset_dir, 'kaggle_house_pred_test.csv')

with urllib.request.urlopen(train_ds_link) as response:
    with open(train_ds_path, 'wb') as file:
        file.write(response.read())

with urllib.request.urlopen(test_ds_link) as response:
    with open(test_ds_path, 'wb') as file:
        file.write(response.read())

## 5.7.2. Kaggle

We'll submit our results to this Kaggle competition: [House Prices: Advanced Regression Techniques | Kaggle](https://www.kaggle.com/c/house-prices-advanced-regression-techniques)

## 5.7.3. Accessing and Reading the Dataset

Let's load our data from the downloaded CSV datasets using [Pandas](https://pandas.pydata.org/) and inspect their shapes. First, we'll need to install Pandas 3.0.2 with `pip`.

In [4]:
%pip install pandas==3.0.2


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Now load the data with [`pandas.read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html#pandas.read_csv) and inspect their shapes.

In [5]:
import pandas as pd

# "df" stands for Pandas DataFrame
train_df = pd.read_csv(train_ds_path)
test_df = pd.read_csv(test_ds_path)
train_df.shape, test_df.shape

((1460, 81), (1459, 80))

The training set has 1460 samples while the test set has 1459 samples. Both sets have 80 input features. Additionally, the training set has 1 output label which is the house price in USD.

Next, let's inspect our data in detail and pre-process it in a form suitable for feeding our deep network for training. As we'll see shortly, not all features are relevant and some data is missing - we'll see how to deal with these in a moment to obtain clean training and test data.

## 5.7.4. Data Preprocessing

TODO